# TrackNetV6 — train B0 (V5-like baseline) on Colab

One-time setup:
1. Upload the dataset archive to Drive: `MyDrive/TrackNetDataset/TrackNetV2.zip`.
2. Create a GitHub fine-grained token with read access to `panditamey/TrackNetV6`
   and add it in Colab **Secrets** (key icon, left sidebar) as `GITHUB_TOKEN`.

Runtime: GPU (A100 or L4 preferred). Frames are decoded to local disk (~40 GB); checkpoints go to Drive.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/TrackNetV6'           # checkpoints + logs
DATA_ZIP = '/content/drive/MyDrive/TrackNetDataset/TrackNetV2.zip'
REPO = 'panditamey/TrackNetV6'
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 1. Clone code (re-run to pull latest)

In [ ]:
import os
token = userdata.get('GITHUB_TOKEN')
url = f'https://{token}@github.com/{REPO}.git'
if os.path.isdir('/content/TrackNetV6/.git'):
    !git -C /content/TrackNetV6 pull -q {url}
else:
    !git clone -q {url} /content/TrackNetV6
    !git -C /content/TrackNetV6 remote set-url origin https://github.com/{REPO}.git  # keep token out of .git/config
%cd /content/TrackNetV6
!git log --oneline -1
!pip -q install -r requirements.txt

## 2. Dataset: unzip from Drive and decode frames once per session (~5-10 min)

In [ ]:
!mkdir -p data && unzip -q -o {DATA_ZIP} -d data/ && rm -rf data/TracknetV2 && mv data/TrackNetV2 data/TracknetV2
!python tools/extract_frames.py --root data/TracknetV2 --out /content/cache/frames_512x288 | tail -3
!du -sh /content/cache

## 3. Train
If the session disconnects, add `--resume {DRIVE}/runs/b0_v5like/last.pt`.

In [ ]:
!python train.py --config config.yaml --set \
    data.cache=/content/cache/frames_512x288 \
    train.out_dir={DRIVE}/runs \
    train.batch_size=8 train.num_workers=8

## 4. Evaluate best checkpoint on the official Test split

In [ ]:
!python evaluate.py --ckpt {DRIVE}/runs/b0_v5like/best.pt --split test --set data.cache=/content/cache/frames_512x288